# Assignment 2



## Question 1

A nuclear fuel pellet is a cylinder, 1.5 cm in lenth and 1 cm in diameter. Assume the surface temperature is 300 C everywhere. Given temperature probe data below, determine the radial temperature profile in the middle of a nuclear fuel pellet (i.e.: T(r, z = 0.75)) using radial basis functions.


In [2]:
import numpy as np

# 20 data points presented in columns: | x | y | z | T |, measurements in cm

data = np.array([
    [5.1690e-02, 2.3766e-01, 6.7059e-01, 5.2645e+02],
    [1.1353e-01, 9.4708e-02, 5.3856e-01, 5.5201e+02],
    [1.6676e-01, 1.4358e-01, 4.6936e-01, 5.0802e+02],
    [1.3610e-01, 3.7207e-02, 2.1694e-01, 4.3663e+02],
    [8.9225e-02, 3.7293e-01, 1.1270e+00, 3.9234e+02],
    [1.9001e-01, 3.7240e-01, 8.4774e-01, 3.8872e+02],
    [5.4849e-02, 3.5425e-01, 5.7478e-01, 4.3784e+02],
    [1.7001e-01, 2.0241e-01, 1.2960e+00, 4.0159e+02],
    [2.0606e-01, 3.1594e-01, 6.4077e-01, 4.2652e+02],
    [2.5382e-01, 2.5859e-01, 4.8610e-01, 4.2481e+02],
    [5.6038e-02, 8.2231e-02, 4.2029e-01, 5.3244e+02],
    [3.1242e-01, 8.0489e-02, 1.1530e+00, 4.2453e+02],
    [6.0186e-02, 4.4891e-01, 3.9941e-01, 3.4207e+02],
    [1.5070e-01, 3.4794e-01, 1.5595e-01, 3.4750e+02],
    [1.8215e-01, 3.4388e-01, 1.0478e+00, 3.9963e+02],
    [1.1633e-01, 4.1011e-01, 5.5001e-01, 3.7611e+02],
    [1.2377e-01, 3.3703e-01, 3.7672e-02, 3.1423e+02],
    [4.6378e-02, 3.3653e-01, 1.4434e+00, 3.2345e+02],
    [2.9063e-02, 3.2584e-02, 2.3977e-01, 4.5993e+02],
    [2.1162e-02, 3.8590e-01, 2.5905e-01, 3.6901e+02]
])

Consider what you know about this system. What extra information do you have in terms of

### a) type(s) of symmetry?

**a) Types of symmetry**

1. **Axial (rotational) symmetry about the cylinder's central axis.** The pellet is a cylinder with a uniform 300 °C surface temperature, so temperature cannot depend on the angle $\theta$. Therefore $T(x, y, z) = T(r, z)$, where $r = \sqrt{x^2 + y^2}$. The 3D problem reduces to a 2D problem in $(r, z)$. This also explains why all probes lie at positive $x$ and $y$: one quadrant fully describes the pellet.

2. **Reflective (mirror) symmetry about the midplane $z = 0.75$ cm.** Both end faces are held at the same temperature, and heat generation is assumed uniform along the length. Therefore $T(r, z) = T(r, 1.5 - z)$. A point measured at $z$ has the same temperature as its mirror point at $1.5 - z$. The profile at $z = 0.75$ is the hottest axial slice.

3. **Radial reflective symmetry through the axis ($r \to -r$).** By axisymmetry, $T(r, z) = T(-r, z)$. The profile is even in $r$, with zero slope at the center: $\left.\frac{\partial T}{\partial r}\right|_{r=0} = 0$. The peak temperature occurs on the centerline.

**How this extra information is used in the RBF fit**

* Convert each $(x, y)$ pair to $r$ and fit in $(r, z)$ only.
* Mirror each point across the midplane, $(r, z) \to (r, 1.5 - z)$, and across the axis, $(r, z) \to (-r, z)$. This increases the 20 data points by up to a factor of 4 and forces the fit to respect the symmetry.
* Add boundary points at $T = 300$ °C along $r = 0.5$ cm, $z = 0$, and $z = 1.5$ cm. These anchor the fit at the surfaces, where there is little probe data.

### b) Boundary conditions?

1. **Dirichlet condition on the outer radial surface:** $T(r = 0.5, z) = 300$ °C for $0 \le z \le 1.5$ cm.

2. **Dirichlet conditions on both end faces:** $T(r, z = 0) = 300$ °C and $T(r, z = 1.5) = 300$ °C for $0 \le r \le 0.5$ cm.

3. **Symmetry (Neumann) condition on the centerline:** $\left.\frac{\partial T}{\partial r}\right|_{r=0} = 0$. Heat cannot cross the axis, so the profile is flat at the center and $T$ is maximum there.

4. **Symmetry (Neumann) condition on the midplane:** $\left.\frac{\partial T}{\partial z}\right|_{z=0.75} = 0$. This follows from the mirror symmetry of the two end faces.

**How these are used in the RBF fit**

* Add synthetic data points at $T = 300$ °C along $r = 0.5$, $z = 0$, and $z = 1.5$. The probe data mostly lies in the interior, so without these points the fit is unconstrained near the surfaces and can extrapolate to nonphysical values.
* Enforce the two Neumann conditions by mirroring the data across $r = 0$ and $z = 0.75$. A fit to symmetric data is itself symmetric, so its slope is automatically zero on those lines.
* All temperatures inside the pellet must be at least 300 °C, since the fuel generates heat and the surface is the coldest point. This provides a quick sanity check on the final profile.

## c) Plot the best guess of the radial temperature profile

Note: RBFs will fail with a linear solver error if two data points exactly overlap.

{Method, implementation, answer, answer}

**c) Method**

1. **Reduce to 2D.** Using axial symmetry, convert each probe location to cylindrical coordinates with $r = \sqrt{x^2 + y^2}$, so $T = T(r, z)$.
2. **Add boundary conditions as data.** Add synthetic points with $T = 300$ °C along the outer surface ($r = 0.5$ cm) and both end faces ($z = 0$ and $z = 1.5$ cm).
3. **Enforce symmetry by mirroring.** Reflect all points (probe and boundary) across the axis, $(r, z) \to (-r, z)$, and across the midplane, $(r, z) \to (r, 1.5 - z)$. The fitted surface is then symmetric, so $\partial T / \partial r = 0$ at $r = 0$ and $\partial T / \partial z = 0$ at $z = 0.75$.
4. **Fit an RBF interpolant.** Use a thin plate spline kernel, $\phi(\rho) = \rho^2 \ln \rho$, which needs no shape parameter and gives the smoothest interpolant. The interpolant is $T(\mathbf{x}) = \sum_i w_i \, \phi(\lVert \mathbf{x} - \mathbf{x}_i \rVert) + p(\mathbf{x})$, where $p$ is a linear polynomial and the weights $w_i$ are found by solving the linear system that matches every data point exactly.
5. **Evaluate and validate.** Evaluate the fit along $z = 0.75$ for $0 \le r \le 0.5$. Check robustness by comparing other kernels and by leave one out cross validation on the 20 probes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RBFInterpolator

R, H, Ts = 0.5, 1.5, 300.0   # radius (cm), length (cm), surface temperature (C)
z_mid = H / 2

# 1. Reduce to (r, z)
r = np.hypot(data[:, 0], data[:, 1])
z = data[:, 2]
T = data[:, 3]

# 2. Boundary condition points at T = 300 C
rb = np.linspace(0, R, 11)
zb = np.linspace(0, H, 16)
bnd = np.vstack([np.c_[rb, np.zeros_like(rb)],        # bottom face z = 0
                 np.c_[rb, np.full_like(rb, H)],      # top face z = 1.5
                 np.c_[np.full_like(zb, R), zb]])     # side wall r = 0.5

def build_set(r, z, T):
    pts = np.vstack([np.c_[r, z], bnd])
    vals = np.r_[T, np.full(len(bnd), Ts)]
    # 3. Mirror across r = 0 and z = 0.75
    P, V = [], []
    for sr in (1, -1):
        for mirror_z in (False, True):
            p = pts.copy()
            p[:, 0] *= sr
            if mirror_z:
                p[:, 1] = H - p[:, 1]
            P.append(p); V.append(vals)
    P, V = np.vstack(P), np.concatenate(V)
    P, idx = np.unique(np.round(P, 8), axis=0, return_index=True)  # remove duplicate points
    return P, V[idx]

# 4. Fit thin plate spline RBF
P, V = build_set(r, z, T)
rbf = RBFInterpolator(P, V, kernel='thin_plate_spline')

# 5. Evaluate along the midplane
r_plot = np.linspace(0, R, 200)
T_plot = rbf(np.c_[r_plot, np.full_like(r_plot, z_mid)])

# Robustness check: other kernels
for k, eps in [('multiquadric', 3), ('inverse_multiquadric', 3), ('gaussian', 3), ('cubic', 1)]:
    Tk = RBFInterpolator(P, V, kernel=k, epsilon=eps)([[0, z_mid]])[0]
    print(f'{k:22s} T(0, 0.75) = {Tk:6.1f} C')

# Leave one out cross validation on the probes
err = []
for i in range(len(T)):
    m = np.arange(len(T)) != i
    Pi, Vi = build_set(r[m], z[m], T[m])
    err.append(RBFInterpolator(Pi, Vi, kernel='thin_plate_spline')([[r[i], z[i]]])[0] - T[i])
print(f'LOO RMS error = {np.sqrt(np.mean(np.square(err))):.2f} C')

for rv in np.linspace(0, R, 6):
    print(f'r = {rv:.1f} cm   T = {rbf([[rv, z_mid]])[0]:6.1f} C')

# Plot
near = np.abs(z - z_mid) < 0.15
plt.figure(figsize=(7, 4.5))
plt.plot(r_plot, T_plot, 'b-', lw=2, label='RBF fit, $z = 0.75$ cm')
plt.scatter(r[near], T[near], c='r', zorder=3, label='Probes with $|z - 0.75| < 0.15$ cm')
plt.axhline(Ts, color='gray', ls='--', lw=1, label='Surface temperature')
plt.xlabel('Radius $r$ (cm)')
plt.ylabel('Temperature (°C)')
plt.title('Radial temperature profile at pellet midplane')
plt.xlim(0, R)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

**c) Answer**

The best estimate of the midplane profile $T(r, z = 0.75)$ is a smooth, roughly parabolic curve. It peaks at about **598 °C on the centerline** and falls to **300 °C at the surface** ($r = 0.5$ cm).

| $r$ (cm) | 0.0 | 0.1 | 0.2 | 0.3 | 0.4 | 0.5 |
|---|---|---|---|---|---|---|
| $T$ (°C) | 598 | 587 | 552 | 492 | 405 | 300 |

The result is robust. Multiquadric, inverse multiquadric, Gaussian, and cubic kernels all give a centerline temperature of 600 to 609 °C, and agree within a few degrees for $r \ge 0.2$ cm. Leave one out cross validation on the 20 probes gives an RMS error of about 1.4 °C. The parabolic shape matches the analytic solution for a cylinder with uniform heat generation, $T(r) = T_s + (T_0 - T_s)\left(1 - r^2/R^2\right)$, and the slope is zero at the centerline as the symmetry condition requires.

# Question 2

You run an experiment and obtain the following data:

| x | y1 | y2 | y3 | y4 | y5 |
|---|---|---|---| --- | --- |
| 0.00 | -29.49 | -2.14 | 15.88 | 22.69 | 28.53 |
| 1.11 | 2.83 | 18.02 | -25.45 | -32.45 | 7.50 |
| 2.22 | 1.97 | -10.49 | -0.18 | -32.10 | -40.31 |
| 3.33 | -38.09 | -46.16 | -7.87 | -33.97 | -38.39 |
| 4.44 | -3.97 | -32.22 | -33.95 | -11.07 | -32.47 |
| 5.56 | 4.45 | -10.88 | 20.43 | 6.57 | -8.49 |
| 6.67 | 50.22 | 51.29 | 80.02 | 66.15 | 84.90 |
| 7.78 | 164.11 | 190.26 | 160.94 | 182.35 | 163.18 |
| 8.89 | 331.75 | 306.51 | 278.40 | 302.13 | 335.44 |
| 10.00 | 517.06 | 483.20 | 476.73 | 512.16 | 500.64 |



In [ ]:
import numpy as np

# Define the table as a list of lists
d = np.array([
    [0.00, -29.49, -2.14, 15.88, 22.69, 28.53],
    [1.11, 2.83, 18.02, -25.45, -32.45, 7.50],
    [2.22, 1.97, -10.49, -0.18, -32.10, -40.31],
    [3.33, -38.09, -46.16, -7.87, -33.97, -38.39],
    [4.44, -3.97, -32.22, -33.95, -11.07, -32.47],
    [5.56, 4.45, -10.88, 20.43, 6.57, -8.49],
    [6.67, 50.22, 51.29, 80.02, 66.15, 84.90],
    [7.78, 164.11, 190.26, 160.94, 182.35, 163.18],
    [8.89, 331.75, 306.51, 278.40, 302.13, 335.44],
    [10.00, 517.06, 483.20, 476.73, 512.16, 500.64]
])

## a) Determine the best cubic polynomial fit to this data with the uncertainty

{method, implementation, answer}

**a) Method**

Model the data as a cubic polynomial:

$$y(x) = a_0 + a_1 x + a_2 x^2 + a_3 x^3$$

1. **Estimate the measurement noise from the replicates.** Each $x$ has 5 repeated measurements, so the sample standard deviation at each $x$ estimates the noise directly, without depending on the model. The standard deviations (13 to 24) show no trend with $x$ or $y$, so the noise is treated as constant. The pooled estimate is $\sigma = \sqrt{\frac{1}{10}\sum_j s_j^2} \approx 18.0$.

2. **Least squares fit.** Fit the mean of the 5 replicates, $\bar{y}_j$, at each $x_j$. Each mean has uncertainty $\sigma_{\bar{y}} = \sigma / \sqrt{5}$. With the Vandermonde matrix $X_{jk} = x_j^k$, the best fit coefficients solve the normal equations:

$$\mathbf{a} = (X^T X)^{-1} X^T \bar{\mathbf{y}}$$

3. **Coefficient uncertainty.** Propagate the measurement noise through the fit with the covariance matrix:

$$\text{Cov}(\mathbf{a}) = \sigma_{\bar{y}}^2 \, (X^T X)^{-1}$$

The uncertainty in each coefficient is the square root of the corresponding diagonal entry. The uncertainty of the fitted curve at any $x$ is $\sigma_y(x) = \sqrt{\mathbf{v}^T \, \text{Cov}(\mathbf{a}) \, \mathbf{v}}$ with $\mathbf{v} = [1, x, x^2, x^3]^T$.

4. **Goodness of fit.** Check the reduced chi squared, $\chi^2_\nu = \frac{1}{N - 4}\sum_j \left(\frac{\bar{y}_j - y(x_j)}{\sigma_{\bar{y}}}\right)^2$. A value near or below 1 means the cubic explains the data within the noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = d[:, 0]
Y = d[:, 1:]              # 5 replicates per x
n_rep = Y.shape[1]
N = len(x)

# 1. Noise estimate from replicates
y_mean = Y.mean(axis=1)
s_j = Y.std(axis=1, ddof=1)
sigma = np.sqrt(np.mean(s_j**2))          # pooled standard deviation
sigma_mean = sigma / np.sqrt(n_rep)       # uncertainty of each mean
print('Replicate std at each x:', np.round(s_j, 1))
print(f'Pooled sigma = {sigma:.2f}')

# 2. Least squares cubic fit
X = np.vander(x, 4, increasing=True)      # columns: 1, x, x^2, x^3
XtX_inv = np.linalg.inv(X.T @ X)
a = XtX_inv @ X.T @ y_mean

# 3. Covariance and coefficient uncertainties
cov = sigma_mean**2 * XtX_inv
a_err = np.sqrt(np.diag(cov))

# 4. Goodness of fit
chi2_nu = np.sum(((y_mean - X @ a) / sigma_mean)**2) / (N - 4)

for k in range(4):
    print(f'a{k} = {a[k]:8.3f} +/- {a_err[k]:.3f}')
print(f'Reduced chi squared = {chi2_nu:.2f}')

# Fitted curve with 95% confidence band
xf = np.linspace(0, 10, 300)
Xf = np.vander(xf, 4, increasing=True)
yf = Xf @ a
yf_err = np.sqrt(np.einsum('ij,jk,ik->i', Xf, cov, Xf))

plt.figure(figsize=(7, 4.5))
plt.scatter(np.repeat(x, n_rep), Y.ravel(), s=15, c='gray', alpha=0.6, label='Measurements')
plt.errorbar(x, y_mean, yerr=sigma_mean, fmt='ko', capsize=3, label='Mean $\\pm \\sigma/\\sqrt{5}$')
plt.plot(xf, yf, 'b-', lw=2, label='Cubic fit')
plt.fill_between(xf, yf - 1.96*yf_err, yf + 1.96*yf_err, color='b', alpha=0.2, label='95% confidence band')
plt.xlabel('$x$')
plt.ylabel('$y$')
plt.title('Cubic least squares fit')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

**a) Answer**

The best cubic fit is:

$$y(x) = (9.5 \pm 7.3) + (-13.1 \pm 6.7)\,x + (-2.6 \pm 1.6)\,x^2 + (0.88 \pm 0.11)\,x^3$$

| Coefficient | Value | Uncertainty ($1\sigma$) |
|---|---|---|
| $a_0$ | 9.53 | 7.32 |
| $a_1$ | −13.09 | 6.71 |
| $a_2$ | −2.59 | 1.61 |
| $a_3$ | 0.881 | 0.106 |

The measurement noise, estimated from the replicates, is $\sigma \approx 18.0$. The reduced chi squared is $\chi^2_\nu \approx 0.38$. This is below 1, so the cubic captures all the structure in the data, and the residuals are fully explained by measurement noise.

The coefficients are strongly correlated. For example, the correlation between $a_2$ and $a_3$ is about $-0.98$. The individual uncertainties therefore overstate the uncertainty of the curve itself, which is tightly constrained (see the confidence band). Any uncertainty propagated from these coefficients should use the full covariance matrix rather than the diagonal alone.

## b) Your manager thinks this should be a quadratic. Which do you think it should be and why?

{Answer}

**Answer**

It should be a **cubic**. Using the measurement noise estimated from the replicates ($\sigma \approx 18$), the quadratic fit has a reduced chi squared of about 10, meaning its residuals are roughly 3 times larger than the noise ($p \approx 10^{-12}$). Its residuals also show a systematic wave pattern, the signature of a missing $x^3$ term. The cubic fit has a reduced chi squared of 0.38, with residuals scattered randomly within the noise, and its cubic coefficient $a_3 = 0.88 \pm 0.11$ is about 8 standard deviations from zero. A quartic does not improve the fit meaningfully (its extra coefficient is consistent with zero and the AIC rises), so the cubic is the simplest model that explains the data. Physically, the data dips slightly before rising steeply, an asymmetric shape a quadratic cannot reproduce.